In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder, MinMaxScaler, RobustScaler, StandardScaler
from sklearn.impute import KNNImputer, SimpleImputer
import warnings
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import miceforest as mf
import sys
from pathlib import Path
import joblib

warnings.filterwarnings('ignore')

import os, json, random, math
import lightgbm as lgb
from tqdm.auto import tqdm

train = pd.read_csv('./open/train.csv')
test  = pd.read_csv('./open/test.csv')

# 컬럼 분리 및 제거
drop_list = ["ID", "불임 원인 - 여성 요인", "불임 원인 - 정자 면역학적 요인"]

category_columns = [
    "시술 당시 나이", "총 시술 횟수", "클리닉 내 총 시술 횟수", "IVF 시술 횟수","DI 시술 횟수", "총 임신 횟수", "IVF 임신 횟수",  "DI 임신 횟수",
    "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수", "시술 시기 코드", "시술 유형", "특정 시술 유형", "배란 자극 여부",  "배란 유도 유형", "단일 배아 이식 여부",
    "착상 전 유전 검사 사용 여부", "착상 전 유전 진단 사용 여부", "남성 주 불임 원인", "남성 부 불임 원인", "여성 주 불임 원인", "여성 부 불임 원인",
    "부부 주 불임 원인","부부 부 불임 원인","불명확 불임 원인", '불임 원인 - 여성 요인',  "불임 원인 - 난관 질환", "불임 원인 - 남성 요인", "불임 원인 - 배란 장애", "불임 원인 - 여성 요인",
    "불임 원인 - 자궁경부 문제",  "불임 원인 - 자궁내막증",  "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인",  "불임 원인 - 정자 운동성", 
    "불임 원인 - 정자 형태",  "배아 생성 주요 이유","난자 출처", "정자 출처", "난자 기증자 나이",  "정자 기증자 나이",
    "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "대리모 여부", "PGD 시술 여부", "PGS 시술 여부"
]

num_columns = [
    '임신 시도 또는 마지막 임신 경과 연수', "난자 채취 경과일",  "난자 해동 경과일",  "난자 혼합 경과일", "배아 이식 경과일", "배아 해동 경과일",
    "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수", "이식된 배아 수", "미세주입 배아 이식 수", "저장된 배아 수",
    "미세주입 후 저장된 배아 수", "해동된 배아 수", "해동 난자 수", "수집된 신선 난자 수", "저장된 신선 난자 수", "혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수",
]

In [ ]:
def drop_inconsistent_thaw_cases(df): 
    cols = ["동결 배아 사용 여부", "해동된 배아 수"] 
    if not set(cols) <= set(df.columns): 
        return df 
    df = df.copy() 
    bad = (df["동결 배아 사용 여부"] != 1) & (df["해동된 배아 수"] > 0) 
    removed = bad.sum() 
    if removed > 0: 
        print(f"Inconsistent thaw rows removed: {removed}") 
        df = df.loc[~bad].reset_index(drop=True) 
        return df

# 특정 column의 범주형 값을 수치형으로 변환하는 함수
def number_mapping(df):
    ivf_cols = [
        "IVF 임신 횟수", "IVF 출산 횟수", "DI 임신 횟수", "DI 출산 횟수",
        "IVF 시술 횟수", "총 임신 횟수", "DI 시술 횟수", "총 시술 횟수", "클리닉 내 총 시술 횟수","총 출산 횟수"
    ]

    mapping = {'0회': 0, '1회': 1, '2회': 2, '3회': 3, '4회': 4, '5회': 5, '6회 이상': 6}

    for col in ivf_cols:
        if col in df.columns:  
            df[col] = df[col].replace(mapping) 
            df[col] = pd.to_numeric(df[col], errors='coerce')  


# 범주형, 수치형 columns들을 list에 추가 및 drop list에 있는 column drop 함수 
def update_column_list(df, category_columns, num_columns, drop_list):
    # 기존 리스트 복사
    category_columns[:] = [col for col in category_columns if col not in drop_list]
    num_columns[:] = [col for col in num_columns if col not in drop_list]

    # df의 모든 컬럼을 검사하여 없던 컬럼을 자동으로 추가
    for col in df.columns:
        if col not in category_columns and col not in num_columns and col not in drop_list:
            if pd.api.types.is_numeric_dtype(df[col]):  # 숫자형 컬럼인지 확인
                num_columns.append(col)
            elif pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):  # 문자형인지 확인
                category_columns.append(col)
                
    df.drop(columns=drop_list, errors='ignore', inplace = True)

# Column 타입을 지정된 타입으로 변환하는 함수
def convert_type(df, category_columns, num_columns, drop_list):
    """
    컬럼 타입을 자동 변환하는 함수.
    """

    sub_category_columns = category_columns.copy()
    sub_num_columns = num_columns.copy()
    
    # _list에 있는 column들은 변환 리스트에서 제외
    sub_category_columns = [x for x in category_columns if x not in drop_list]
    sub_num_columns = [x for x in num_columns if x not in drop_list]

    # 컬럼 타입 변환
    for col in df.columns:
        if col in sub_category_columns:
            # 기존 dtype이 category가 아닌 경우 category로 변환
            if not pd.api.types.is_categorical_dtype(df[col]):
                df[col] = df[col].astype(str).astype('category')
                # sub_num_columns.append(col)  # category가 아니었으므로 숫자 컬럼 리스트에 추가

        elif col in sub_num_columns:
            # 기존 dtype이 numeric이 아닌 경우 float로 변환
            if not pd.api.types.is_numeric_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], errors='coerce')
                # sub_category_columns.append(col)  # 숫자가 아니었으므로 category 컬럼 리스트에 추가

# Catboost를 위해 category 함수르 string으로 변환
def convert_category_columns(df, category_columns):
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype(str)  # 모든 값을 문자열로 변환

# Mice model이 결측치를 채우기 전 전처리
def pre_mice(df):
    df["배아 해동 여부"] = df["배아 해동 경과일"].notna().astype(int)
    df["배란 유도_배란 자극"] = df["배란 자극 여부"].astype(str) + "_" + df["배란 유도 유형"].astype(str)

# 시술 유형이 DI인 경우, 특정 column들 결측치 처리 함수
def pp_fillna_di(df):
    di_nan_col = [
        '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
        '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
        '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수',
        '기증자 정자와 혼합된 난자 수', "난자 채취 경과일", "난자 해동 경과일", "배아 이식 경과일", "배아 해동 경과일",
        "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "단일 배아 이식 여부", '난자 혼합 경과일',
        '임신 시도 또는 마지막 임신 경과 연수', 
    ]

    row = df['시술 유형'] == "DI"
    
    df.loc[row, di_nan_col] = df.loc[row, di_nan_col].fillna(0)


# 비율 및 수치 관련 전처리 함수
def pp_ratio(df):
    df['난자 성공률'] = np.where(df['수집된 신선 난자 수'] == 0, -1, df['미세주입된 난자 수'] / df['수집된 신선 난자 수'])
    df['저장된 배아 사용률'] = np.where(df['총 생성 배아 수'] + df['해동된 배아 수'] == 0, -1, df['저장된 배아 수'] / (df['총 생성 배아 수'] + df['해동된 배아 수']))

    # 시술 횟수 관련 전처리
    df['IVF 임신률'] = np.where(df['IVF 시술 횟수'] == 0, -1, df['IVF 임신 횟수'] / df['IVF 시술 횟수'])
    df['DI 임신률'] = np.where(df['DI 시술 횟수'] == 0, -1, df['DI 임신 횟수'] / df['DI 시술 횟수'])

    df['IVF 출산률'] = np.where(df['IVF 임신 횟수'] == 0, -1, df['IVF 출산 횟수'] / df['IVF 임신 횟수'])
    df['DI 출산률'] = np.where(df['DI 임신 횟수'] == 0, -1, df['DI 출산 횟수'] / df['DI 임신 횟수'])

    df['IVF 시술 비율'] = np.where((df['IVF 시술 횟수'] + df['DI 시술 횟수']) == 0, -1, 
                                df['IVF 시술 횟수'] / (df['IVF 시술 횟수'] + df['DI 시술 횟수']))

    df['총 출산률'] = np.where(df['총 임신 횟수'] == 0, -1, df['총 출산 횟수'] / df['총 임신 횟수'])

    df["클리닉 시술 비율"] = np.where((df["IVF 시술 횟수"] + df["DI 시술 횟수"]) == 0, -1, 
                                df["클리닉 내 총 시술 횟수"] / (df["IVF 시술 횟수"] + df["DI 시술 횟수"]))
    
    df['미세주입 배아 이식률'] = np.where((df['해동된 배아 수'] + df['미세주입에서 생성된 배아 수'] - df['미세주입 후 저장된 배아 수']) == 0, -1, 
                                    df['미세주입 배아 이식 수'] / (df['해동된 배아 수'] + df['미세주입에서 생성된 배아 수'] - df['미세주입 후 저장된 배아 수']))

    df['이식된 배아 수 대비 미세주입 배아 수'] = np.where(df['이식된 배아 수'] == 0, -1, 
                                                df['미세주입 배아 이식 수'] / df['이식된 배아 수'])

    df['미세주입 배아 생성률'] = np.where(df['미세주입된 난자 수'] == 0, -1, 
                                    df['미세주입에서 생성된 배아 수'] / df['미세주입된 난자 수'])

    df['혼합 대비 생성 배아률'] = np.where((df['혼합된 난자 수'] + df['해동 난자 수']) == 0, -1, 
                                    df['총 생성 배아 수'] / (df['혼합된 난자 수'] + df['해동 난자 수']))

    df['신선 난자 선택'] = np.where((df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']) == 0, -1, 
                            df['혼합된 난자 수'] / (df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']))

    df['수집 난자 이식률'] = np.where((df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']) == 0, -1, 
                                df['이식된 배아 수'] / (df['수집된 신선 난자 수'] - df['저장된 신선 난자 수']))

    df['혼합 대비 미세주입 난자 비율'] = np.where(df['혼합된 난자 수'] == 0, -1, 
                                            df['미세주입된 난자 수'] / df['혼합된 난자 수'])

    df['경과일 대비 배아 이식 수'] = np.where(df['배아 이식 경과일'] == 0, -1, 
                                df['이식된 배아 수'] / df['배아 이식 경과일'])

    df['총 배아 사용 대비 이식'] = np.where((df['해동된 배아 수'] + df['총 생성 배아 수'] + df['저장된 배아 수']) == 0, -1, 
                                            df['이식된 배아 수'] / (df['해동된 배아 수'] + df['총 생성 배아 수'] + df['저장된 배아 수']))

    df['총 사용 배아'] = df['해동된 배아 수'] + df['총 생성 배아 수']
    df["IVF 정자 미혼합 여부"] = ((df["파트너 정자와 혼합된 난자 수"] == 0) & (df["기증자 정자와 혼합된 난자 수"] == 0) & (df["시술 유형"] == "IVF")).astype(int)
    
    df['대리모 여부'] = df['대리모 여부'].fillna(-1)
    df['배란 유도 유형'].replace({'생식선 자극 호르몬' : '기록되지 않은 시행', '세트로타이드 (억제제)': '기록되지 않은 시행'} , inplace = True)

    # 시술 시기에 따른 각 column들의 영향
    df['시기별 배아 사용 여부'] = (
            df['시술 시기 코드'].str.lower()
            + df['동결 배아 사용 여부'].fillna(0).astype(int).astype(str)
            + df['신선 배아 사용 여부'].fillna(0).astype(int).astype(str)
            + df['기증 배아 사용 여부'].fillna(0).astype(int).astype(str)
        ).astype('category')
    
    # 신선 배아 사용시 시기에 따른 난자 개수의 영향 반영
    df['과배란 유도 주사 효과'] = 'frozen'
    fresh = (df['신선 배아 사용 여부'] == 1) & (df['동결 배아 사용 여부'] == 0)

    mask2 = (df['수집된 신선 난자 수'] > 10) & fresh
    mask3 = (df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 10) & fresh

    df.loc[mask2, '과배란 유도 주사 효과'] = df.loc[mask2, '시술 시기 코드'] + '_1'
    df.loc[mask3, '과배란 유도 주사 효과'] = df.loc[mask3, '시술 시기 코드'] + '_2'

    # 동결 배아 사용시에 시기별 해동 배아의 수 반영 
    only_frozen = (df['동결 배아 사용 여부'] == 1) &  (df['신선 배아 사용 여부'] == 0)

    df['시기별 해동 배아 영향'] = 'frozen'

    mask2 = (df['해동된 배아 수'] > 3) & only_frozen
    mask3 = (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 3) & only_frozen

    df.loc[mask2, '시기별 해동 배아 영향'] = df.loc[mask2, '시술 시기 코드'] + '_1'
    df.loc[mask3, '시기별 해동 배아 영향'] = df.loc[mask3, '시술 시기 코드'] + '_2'

    # 신선/동결 배아 사용 시 시기별 난자 및 배아의 수 반영 
    df['시기별 배아 영향'] = 'both'
    both = (df['신선 배아 사용 여부'] == 1) & (df['동결 배아 사용 여부'] == 1)

    mask2 = ((df['수집된 신선 난자 수'] > 7) & (df['해동된 배아 수'] > 2)) & both
    mask3 = ((df['수집된 신선 난자 수'] > 7) & (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 2)) & both
    mask4 = ((df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 7) & (df['해동된 배아 수'] > 2)) & both
    mask5 = ((df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 7) & (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 2)) & both
    mask6 = (df['수집된 신선 난자 수'] == 0) & (df['해동된 배아 수'] > 2) & both
    mask7 = (df['수집된 신선 난자 수'] == 0) & (df['해동된 배아 수'] > 0) & (df['해동된 배아 수'] <= 2) & both
    mask8 = (df['수집된 신선 난자 수'] > 7) & (df['해동된 배아 수'] == 0) & both
    mask9 = ((df['수집된 신선 난자 수'] > 0) & (df['수집된 신선 난자 수'] <= 7)) & (df['해동된 배아 수'] == 0) & both

    df.loc[mask2, '시기별 배아 영향'] = df.loc[mask2, '시술 시기 코드'] + '_1'
    df.loc[mask3, '시기별 배아 영향'] = df.loc[mask3, '시술 시기 코드'] + '_2'
    df.loc[mask4, '시기별 배아 영향'] = df.loc[mask4, '시술 시기 코드'] + '_3'
    df.loc[mask5, '시기별 배아 영향'] = df.loc[mask5, '시술 시기 코드'] + '_4'
    df.loc[mask6, '시기별 배아 영향'] = df.loc[mask6, '시술 시기 코드'] + '_5'
    df.loc[mask7, '시기별 배아 영향'] = df.loc[mask7, '시술 시기 코드'] + '_6'
    df.loc[mask8, '시기별 배아 영향'] = df.loc[mask8, '시술 시기 코드'] + '_7'
    df.loc[mask9, '시기별 배아 영향'] = df.loc[mask9, '시술 시기 코드'] + '_8'

    # 이상적인 배양 기간 설정
    mask = (df['배아 이식 경과일'].isin([3, 5]))
    df['이상적인 배양 기간'] = 'NO'
    df.loc[mask, '이상적인 배양 기간'] = 'YES'

    # 시기별 각 column들의 영향 정리
    df['시기별 단일 배아 이식 여부'] = (df['시술 시기 코드'].str.lower() + df['단일 배아 이식 여부'].astype(int).astype(str))
    df['시기별 유전 진단 사용 여부'] = df['시술 시기 코드'] + df['착상 전 유전 진단 사용 여부'].fillna(0).astype(int).astype(str)
    df['시기별 배란 자극 사용 여부'] = df['시술 시기 코드'] + df['배란 자극 여부'].fillna(0).astype(int).astype(str)
    
    # 신선 배아의 배양 시간에 따른 영향
    if all(col in df.columns for col in ["배아 이식 경과일", "난자 혼합 경과일"]):
        df["신선 배아 배양 시간"] = df["배아 이식 경과일"] - df["난자 혼합 경과일"]
        
    df["신선 배아 배양 시간"] = df["신선 배아 배양 시간"].fillna(0)

    # 동결 배아의 배양 시간에 따른 영향 
    if all(col in df.columns for col in ["배아 이식 경과일", "배아 해동 경과일"]):
        df["동결 배아 배양 시간"] = df["배아 이식 경과일"] - df["배아 해동 경과일"]
        
    df["동결 배아 배양 시간"] = df["동결 배아 배양 시간"].fillna(0)


# 시술 당시 나이가 알 수 없음인 경우 만45-50세로 대체
def pp_age_replace(df): 
    age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세']
    df['시술 당시 나이'] = df['시술 당시 나이'].replace('알 수 없음', '만43-44세')


# PGD, PGS의 영향 전처리 함수수
def pp_merge_pgd_pgs(df):
    col = ["착상 전 유전 검사 사용 여부", "착상 전 유전 진단 사용 여부", "PGD 시술 여부", "PGS 시술 여부"]
    for i in col:
        df[col] = df[col].fillna(0)
    
    df["과거 검사 사용 여부"] = df["착상 전 유전 검사 사용 여부"] + df["착상 전 유전 진단 사용 여부"]
    df["현재 검사 사용 여부"] = df["PGD 시술 여부"] + df["PGS 시술 여부"]

    df["과거 PGD 비율"] = np.where(df["과거 검사 사용 여부"] == 0, -1, df["착상 전 유전 검사 사용 여부"] / df["과거 검사 사용 여부"])
    df["현재 PGD 비율"] = np.where(df["현재 검사 사용 여부"] == 0, -1, df["PGD 시술 여부"] / df["현재 검사 사용 여부"])
    
    for i in col:
        df[i] = df[i].fillna(0)
        df[i] = df[i].astype('category')


# 각 기증자 나이의 알 수 없음 전처리
def pp_donate_egg_sperm(df): # 임의 
    col = ["난자 기증자 나이", "정자 기증자 나이"]
    
    for i in col:
        df[i].replace({'알 수 없음' : '만21-25세'} , inplace = True)


# 배아 생성 주요 이유의 값 정리
def pp_categorize_embryo_creation_reason(df):
    col= "배아 생성 주요 이유"
    df[col] = df[col].fillna("시술용")  # NaN은 기본적으로 '시술용'으로 설정

    category_map = {
        "현재 시술용": "시술용",
        "배아 저장용, 현재 시술용": "시술용",
        "난자 저장용, 현재 시술용" : "시술용",
        "현재 시술용" : "시술용",

        "기증용": "기증용",
        "기증용, 현재 시술용": "기증용",
        "기증용, 배아 저장용": "기증용",
        "기증용, 난자 저장용": "기증용",

        "배아 저장용": "배아 저장용",

        "난자 저장용": "난자 저장용",
        "난자 저장용, 배아 저장용": "난자 저장용",

        "난자 저장용, 배아 저장용, 연구용": "연구용",
        "연구용, 현재 시술용": "연구용",

        "기증용, 배아 저장용, 현재 시술용": "복합용도"
    }

    df[col] = df[col].replace(category_map)


# 난자의 나이를 계산하는 함수
def pp_egg_age(df):
    if "난자 출처" not in df.columns:
        return df
    df["난자 나이"] = "알 수 없음"
    if "시술 당시 나이" in df.columns:
        df.loc[df["난자 출처"] == "본인 제공", "난자 나이"] = df["시술 당시 나이"]
    if "난자 기증자 나이" in df.columns:
        mask = (df["난자 출처"] == "기증 제공") & (df["난자 기증자 나이"] != "알 수 없음")
        df.loc[mask, "난자 나이"] = df.loc[mask, "난자 기증자 나이"]
        mask = (df["난자 출처"] == "기증 제공") & (df["난자 기증자 나이"] == "알 수 없음")
        df.loc[mask, "난자 나이"] = "만18-34세"
    if "시술 당시 나이" in df.columns:
        df.loc[df["난자 출처"] == "알 수 없음", "난자 나이"] = df["시술 당시 나이"]
    return df

# 불임 원인 colunms 전처리 함수
def pp_merge_fail_causes(df):
    male_fail = ["불임 원인 - 남성 요인", "불임 원인 - 정자 농도", "불임 원인 - 정자 면역학적 요인",
                "불임 원인 - 정자 운동성", "불임 원인 - 정자 형태"]
    
    female_fail = ["불임 원인 - 난관 질환", "불임 원인 - 배란 장애", "불임 원인 - 여성 요인", 
                    "불임 원인 - 자궁경부 문제", "불임 원인 - 자궁내막증"]
    
    df["남성 불임 심각도"] = df[male_fail].sum(axis = 1)
    df["여성 불임 심각도"] = df[female_fail].sum(axis = 1)
    
    df["여성 심각도"] = np.where((df['남성 불임 심각도'] + df["여성 불임 심각도"]) == 0, -1, df["여성 불임 심각도"] / (df['남성 불임 심각도'] + df["여성 불임 심각도"]))


# 모든 전처리 함수를 처리하는 통합 전처리 함수
def preprocess(train, test, category_columns, num_columns, drop_list, mdl):    
    mice_col_lst = [
        '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
        '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
        '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수',
        '기증자 정자와 혼합된 난자 수', "난자 채취 경과일", "난자 해동 경과일", "배아 이식 경과일", "배아 해동 경과일",
        "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "단일 배아 이식 여부", '난자 혼합 경과일',
        '임신 시도 또는 마지막 임신 경과 연수'
    ]

    # Mice 전 전처리 함수 및 시술 유형 DI 관련 결측치 리리
    for df in [train, test]:
        pre_mice(df)
        pp_fillna_di(df)
    
    # Mice : mice_col_list 중 결측치 채우기
    train, test = mice_process(train, test, mice_col_lst, mdl)

    for df in [train, test]:
        drop_inconsistent_thaw_cases(df)
        # 횟수(IVF) 문자열로 되어있는 것을 number로 mapping
        number_mapping(df)

        # 배아 주요 생성 이유를 Mapping
        pp_categorize_embryo_creation_reason(df)

        # 비율 전처리  
        pp_ratio(df)
        
        # 시술 당시 나이를 만45-50세 대체.
        pp_age_replace(df)

        # PGS, PGD 여부를 merge
        pp_merge_pgd_pgs(df)

        # 난자와 정자 기증자 나이 대체. 
        pp_donate_egg_sperm(df)

        # 난자 나이 계산
        pp_egg_age(df)

        # 불임 원인 전처리 
        pp_merge_fail_causes(df)

    # number, category column에 따라 Dataframe 변환
    for df in [train, test]:
        convert_type(df, category_columns, num_columns, drop_list)
        update_column_list(df, category_columns, num_columns, drop_list)
        convert_category_columns(df, category_columns)

    return train, test

# Mice model training 함수
def training_mice(df, target_columns):
    df_subset = df[target_columns].copy()
    
    mdl = mf.ImputationKernel(df_subset, random_state=42)
    mdl.mice(iterations=5, n_estimators=50, n_jobs=-1)

    return mdl

# Mice model을 train 또는 test 데이터 셋에 적용하는 함수
def apply_mice(df, target_columns, mdl, isTraining):
    df_subset = df[target_columns].copy()

    if isTraining:
        # Mice model로부터 train 데이터 결측치 추정
        imputed_df =  mdl.complete_data()
        print('Training dataset nan filled')
    else:
        # Train data로 학습된 Mice model로부터 test 데이터 결측치 생성  
        imputed_df = mdl.impute_new_data(df_subset).complete_data()
        print('Test dataset nan filled')

    df[target_columns] = imputed_df[target_columns]

    return df

# Mice 모델을 train, test에 적용하는 함수. 
def mice_process(train, test, target_columns, mdl):
    print("MICE list : ", target_columns)
    
    train = apply_mice(train, target_columns, mdl, True)
    test = apply_mice(test, target_columns, mdl, False)

    return train, test

train = pd.read_csv('./open/train.csv')

# Mice로 채우고자 하는 column list
target_columns = [
    '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
    '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
    '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수',
    '기증자 정자와 혼합된 난자 수', "난자 채취 경과일", "난자 해동 경과일", "배아 이식 경과일", "배아 해동 경과일",
    "동결 배아 사용 여부", "신선 배아 사용 여부", "기증 배아 사용 여부", "단일 배아 이식 여부", '난자 혼합 경과일',
    '임신 시도 또는 마지막 임신 경과 연수'
]

pp_fillna_di(train)

mdl = training_mice(train, target_columns)

# # Mice model 저장
# joblib.dump(mdl, "./kds_model.pkl")
# print("mice 모델 학습 후 저장 완료")
# # 학습용 코드

mdl = joblib.load("./kds_model.pkl") # mice 로드
print("mice 모델 로드 완료")

train, test = preprocess(train, test, category_columns, num_columns, drop_list, mdl)

mice 모델 로드 완료
MICE list :  ['미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수', '난자 채취 경과일', '난자 해동 경과일', '배아 이식 경과일', '배아 해동 경과일', '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부', '단일 배아 이식 여부', '난자 혼합 경과일', '임신 시도 또는 마지막 임신 경과 연수']
Training dataset nan filled
Test dataset nan filled
Inconsistent thaw rows removed: 259
Inconsistent thaw rows removed: 91


In [ ]:
# 나이 패널티
def age_penalty(df):
    df = df.copy()
    mapping = {
        "만18-34세": 0,
        "만35-37세": 1,
        "만38-39세": 2,
        "만40-42세": 3,
        "만43-44세": 4,
        "만45-50세": 5,
    }

    df["age_penalty"] = df["시술 당시 나이"].map(mapping).fillna(0)
    return df

def ivf_donor_age_features(df):
    df = df.copy()
    if "난자 기증자 나이" in df.columns:
        donor_age_map = {
            "만20세 이하": 0,
            "만21-25세": 1,
            "만26-30세": 2,
            "만31-35세": 3,
        }
        df["난자기증자나이_ord"] = df["난자 기증자 나이"].map(donor_age_map)
    if "정자 기증자 나이" in df.columns:
        sperm_age_map = {
            "만20세 이하": 0,
            "만21-25세": 1,
            "만26-30세": 2,
            "만31-35세": 3,
            "만36-40세": 4,
            "만41-45세": 5,
        }
        df["정자기증자나이_ord"] = df["정자 기증자 나이"].map(sperm_age_map)
    return df

def make_egg_age_ord(df):
    df = df.copy()
    if "난자 나이" not in df.columns:
        return df
    age_map = {
        "만20세 이하": 0,
        "만21-25세": 0,
        "만26-30세": 0,
        "만31-35세": 0,

        "만35-37세": 1,
        "만38-39세": 2,
        "만40-42세": 3,
        "만43-44세": 4,
        "만45-50세": 5,
    }
    df["난자나이_ord"] = df["난자 나이"].map(age_map)
    return df

def transfer_power(df):
    df = df.copy()
    if {"이식된 배아 수", "age_penalty"} <= set(df.columns):
        df["이식 공격성"] = df["이식된 배아 수"] / (df["age_penalty"] + 1)
    if {"이식된 배아 수", "난자나이_ord"} <= set(df.columns):
        df["이식 공격성2"] = df["이식된 배아 수"].astype(str) + "_" + df["난자나이_ord"].astype(str)
    return df

def embryo_margin(df):
    df = df.copy()
    if {"저장된 배아 수", "이식된 배아 수"} <= set(df.columns):
        df["배아 여유도"] = df["저장된 배아 수"] - df["이식된 배아 수"]
    return df

def embryo_efficiency(df):
    df = df.copy()
    if {"총 생성 배아 수", "수집된 신선 난자 수"} <= set(df.columns):
        df["배아 생성 효율"] = df["총 생성 배아 수"] / (df["수집된 신선 난자 수"] + 1)
    return df

def age_strategy(df):
    df = df.copy()
    if {"age_penalty", "이식된 배아 수"} <= set(df.columns):
        df["고령 공격 전략"] = df["age_penalty"] * df["이식된 배아 수"]
    return df

def time_pressure(df):
    df = df.copy()
    if {"배아 이식 경과일", "저장된 배아 수"} <= set(df.columns):
        df["시간 압박 지수"] = df["저장된 배아 수"] / (df["배아 이식 경과일"] + 1)
    return df

def risk_group_features(df):
    if "총 생성 배아 수" in df.columns:
        df["배아풍부"] = (df["총 생성 배아 수"] >= 5).astype(int)
        df["배아부족"] = (df["총 생성 배아 수"] <= 2).astype(int)
    return df

def preprocess_add(df):
    df = age_penalty(df)
    df = ivf_donor_age_features(df)
    df = transfer_power(df)
    df = embryo_margin(df)
    df = embryo_efficiency(df)
    df = age_strategy(df)
    df = risk_group_features(df)
    df = time_pressure(df)
    return df

train = preprocess_add(train)
test  = preprocess_add(test)

print("데이터 전처리 완료:")
print("train:", train.shape)
print("test:", test.shape)

데이터 전처리 완료:
train: (256351, 128)
test: (90067, 127)


In [34]:
vc = train["배아수_구간"].value_counts(dropna=False)

print(vc)

배아수_구간
2개      110845
1개       93791
없음       42835
3개이상      8880
Name: count, dtype: int64


In [33]:
# =============================
# CatBoost 25-Fold CV + Test 제출용
# =============================
import os
import glob
from datetime import datetime
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score,
    f1_score, roc_auc_score
)

# =============================
# 설정
# =============================
TARGET = "임신 성공 여부"
SAVE_DIR = "./models"
os.makedirs(SAVE_DIR, exist_ok=True)

k_fold = 25
random_state = 2024
THRESHOLD = 0.5

params = {
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'iterations': 3000,
    'depth': 7,
    'learning_rate': 0.059442066023287284,
    'l2_leaf_reg': 5.916099093406194,
    'bagging_temperature': 0.0021443688183697563,
    'random_strength': 0.989238115265615,
    'od_type': 'Iter',
    'od_wait': 200,
    'random_seed': random_state,
    'use_best_model': True,
    'class_weights': {0: 1.0, 1: 1.004614669263528},
    'task_type': 'CPU',
    'thread_count': -1,
    'verbose': 100
}

# =============================
# CatBoost용 전처리
# =============================
def finalize_for_catboost(df, target=None):
    df = df.copy()
    for col in df.columns:
        if col == target:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(str).fillna("Unknown")
    return df

train = finalize_for_catboost(train, TARGET)
test = finalize_for_catboost(test, TARGET)

# =============================
# X, y 준비
# =============================
y = train[TARGET].values
X_ivf = train.drop(columns=[TARGET])
ivf_t = test.copy()

# categorical index
cat_features = [
    X_ivf.columns.get_loc(c) for c in X_ivf.select_dtypes(include=['object']).columns.tolist()
]

# =============================
# Metrics 함수
# =============================
def eval_metrics_append(metrics_dict, y_true, pred, pred_proba):
    metrics_dict['confusion'].append(confusion_matrix(y_true, pred))
    metrics_dict['accuracy'].append(accuracy_score(y_true, pred))
    metrics_dict['precision'].append(precision_score(y_true, pred))
    metrics_dict['recall'].append(recall_score(y_true, pred))
    metrics_dict['f1_score'].append(f1_score(y_true, pred))
    metrics_dict['roc_auc'].append(roc_auc_score(y_true, pred_proba))

eval_metrics_dict = {k: [] for k in ['confusion','accuracy','precision','recall','f1_score','roc_auc']}
oof_pred = np.zeros(len(y))
test_pred_lst = []

# =============================
# KFold CV
# =============================
skf = StratifiedKFold(n_splits=k_fold, shuffle=True, random_state=random_state)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_ivf, y)):
    print(f"\n========== Fold {fold+1} ==========")
    x_train, y_train = X_ivf.iloc[train_idx], y[train_idx]
    x_val, y_val = X_ivf.iloc[val_idx], y[val_idx]

    train_pool = Pool(x_train, y_train, cat_features=cat_features)
    val_pool   = Pool(x_val, y_val, cat_features=cat_features)

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=250)

    # validation
    val_pred_proba = model.predict_proba(x_val)[:, 1]
    val_pred = (val_pred_proba >= THRESHOLD).astype(int)
    eval_metrics_append(eval_metrics_dict, y_val, val_pred, val_pred_proba)

    # OOF 채우기
    oof_pred[val_idx] = val_pred_proba

    # test 예측
    test_pred_lst.append(model.predict_proba(ivf_t)[:, 1])

    # 모델 저장
    fold_save_path = os.path.join(SAVE_DIR, f"IVF_Fold{fold+1}.cbm")
    model.save_model(fold_save_path)
    print(f"Fold {fold+1} 모델 저장 완료: {fold_save_path}")

# =============================
# CV 결과 출력
# =============================
print("\n========== CV RESULT ==========")
print("Mean Accuracy:", np.mean(eval_metrics_dict['accuracy']))
print("Mean AUC:", np.mean(eval_metrics_dict['roc_auc']))
print("Mean F1 :", np.mean(eval_metrics_dict['f1_score']))
print("Mean Precision:", np.mean(eval_metrics_dict['precision']))
print("Mean Recall:", np.mean(eval_metrics_dict['recall']))

# =============================
# OOF & Test 저장
# =============================
np.save(os.path.join(SAVE_DIR, f"catboost_oof.npy"), oof_pred)
test_pred_final = np.mean(test_pred_lst, axis=0)
np.save(os.path.join(SAVE_DIR, f"IVF_test_pred_mean.npy"), test_pred_final)

# =============================
# 제출용 CSV 생성
# =============================
submission = pd.read_csv("./open/sample_submission.csv")
submission.iloc[:, 1] = test_pred_final  # 확률 그대로 넣기
submission_file = f'optimus_IVF_{datetime.now().strftime("%m%d-%H%M")}.csv'
submission.to_csv(submission_file, index=False)
print(f"\n✅ 제출 파일 생성 완료: {submission_file}")



========== Fold 1 ==========
0:	learn: 0.6618823	test: 0.6618852	best: 0.6618852 (0)	total: 352ms	remaining: 17m 35s
100:	learn: 0.4878764	test: 0.4877985	best: 0.4877985 (100)	total: 22.3s	remaining: 10m 41s
200:	learn: 0.4855808	test: 0.4865183	best: 0.4865183 (200)	total: 43.8s	remaining: 10m 9s
300:	learn: 0.4840288	test: 0.4861753	best: 0.4861625 (288)	total: 1m 6s	remaining: 9m 51s
400:	learn: 0.4826446	test: 0.4860518	best: 0.4859860 (378)	total: 1m 29s	remaining: 9m 37s
500:	learn: 0.4814630	test: 0.4859030	best: 0.4858787 (486)	total: 1m 51s	remaining: 9m 15s
600:	learn: 0.4802420	test: 0.4859330	best: 0.4858787 (486)	total: 2m 14s	remaining: 8m 55s
700:	learn: 0.4791546	test: 0.4858851	best: 0.4858402 (663)	total: 2m 36s	remaining: 8m 34s
800:	learn: 0.4780920	test: 0.4859273	best: 0.4858402 (663)	total: 2m 58s	remaining: 8m 10s
900:	learn: 0.4770040	test: 0.4859926	best: 0.4858402 (663)	total: 3m 21s	remaining: 7m 48s
Stopped by overfitting detector  (250 iterations wait)



In [16]:
# =============================
# Feature Importance
# =============================
def get_feature_importance(model, X, top_n=50):
    import pandas as pd

    fi = pd.DataFrame({
        "feature": X.columns,
        "importance": model.get_feature_importance(Pool(X, cat_features=cat_features))
    }).sort_values("importance", ascending=False)

    return fi.head(top_n), fi

# 예시: 마지막 fold 모델 사용
# for문 안에서 마지막 fold 모델이 'model'에 남아있음
top_features, full_fi = get_feature_importance(model, X_ivf)

print("\n===== Top Features =====")
print(top_features)


===== Top Features =====
                   feature  importance
112                 배아수_구간   13.776164
108                 이식 공격성   13.178981
77   이식된 배아 수 대비 미세주입 배아 수   11.823657
39                이식된 배아 수    9.036972
111               고령 공격 전략    5.922558
83          경과일 대비 배아 이식 수    3.933469
101                  난자 나이    2.731284
105            age_penalty    2.501253
84           총 배아 사용 대비 이식    2.457691
68              저장된 배아 사용률    2.009605
25             배아 생성 주요 이유    1.846807
95             신선 배아 배양 시간    1.816131
45             수집된 신선 난자 수    1.710600
66             배란 유도_배란 자극    1.567641
51                   정자 출처    1.528021
1                 시술 당시 나이    1.459420
63               배아 이식 경과일    1.411066
41                저장된 배아 수    1.204291
81               수집 난자 이식률    1.088802
80                신선 난자 선택    1.083434
85                 총 사용 배아    0.971708
50                   난자 출처    0.899446
79            혼합 대비 생성 배아률    0.868936
119               시간 압박 지수    0.833960

In [18]:
def drop_useless_columns_relative(df, importance_df, quantile=0.1):
    threshold = importance_df["importance"].quantile(quantile)
    drop_cols = importance_df.loc[
        importance_df["importance"] < threshold, "feature"
    ].tolist()

    print(f"Drop {len(drop_cols)} columns")
    return df.drop(columns=drop_cols, errors="ignore"), drop_cols

train_reduced, dropped_cols = drop_useless_columns_relative(
    train, full_fi, quantile=0.1
)

test_reduced = test.drop(columns=dropped_cols, errors="ignore")

print(train_reduced.shape, test_reduced.shape)

Drop 12 columns
(256351, 109) (90067, 108)


In [20]:
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from datetime import datetime

def run_catboost_cv(
    train_df, test_df, target,
    cat_features=None, n_splits=25, threshold=0.5,
    save_dir="./models", random_state=2024, verbose=100
):
    os.makedirs(save_dir, exist_ok=True)

    # ==========================
    # CatBoost 전처리
    # ==========================
    train = train_df.copy()
    test = test_df.copy()
    
    for col in train.columns:
        if col != target and not pd.api.types.is_numeric_dtype(train[col]):
            train[col] = train[col].astype(str).fillna("Unknown")
            test[col]  = test[col].astype(str).fillna("Unknown")
    
    X = train.drop(columns=[target])
    y = train[target].values
    X_test = test.copy()

    if cat_features is None:
        cat_features = [X.columns.get_loc(c) for c in X.select_dtypes(include=['object']).columns]

    # ==========================
    # Evaluation 기록용
    # ==========================
    eval_metrics_dict = {k: [] for k in ['confusion','accuracy','precision','recall','f1_score','roc_auc']}
    oof_pred = np.zeros(len(y))
    test_pred_list = []

    # ==========================
    # KFold
    # ==========================
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n========== Fold {fold+1} ==========")

        x_train, y_train = X.iloc[train_idx], y[train_idx]
        x_val, y_val = X.iloc[val_idx], y[val_idx]

        train_pool = Pool(x_train, y_train, cat_features=cat_features)
        val_pool   = Pool(x_val, y_val, cat_features=cat_features)

        model = CatBoostClassifier(
            iterations=3000,
            depth=7,
            learning_rate=0.059442066023287284,
            loss_function='Logloss',
            eval_metric='Logloss',
            random_seed=random_state,
            use_best_model=True,
            verbose=verbose
        )

        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=250)

        # OOF
        val_pred_proba = model.predict_proba(x_val)[:, 1]
        val_pred = (val_pred_proba >= threshold).astype(int)
        oof_pred[val_idx] = val_pred_proba

        # 평가 기록
        eval_metrics_dict['confusion'].append(confusion_matrix(y_val, val_pred))
        eval_metrics_dict['accuracy'].append(accuracy_score(y_val, val_pred))
        eval_metrics_dict['precision'].append(precision_score(y_val, val_pred))
        eval_metrics_dict['recall'].append(recall_score(y_val, val_pred))
        eval_metrics_dict['f1_score'].append(f1_score(y_val, val_pred))
        eval_metrics_dict['roc_auc'].append(roc_auc_score(y_val, val_pred_proba))

        # Test
        test_pred_list.append(model.predict_proba(X_test)[:, 1])

        # 모델 저장
        fold_save_path = os.path.join(save_dir, f"CatBoost_Fold{fold+1}.cbm")
        model.save_model(fold_save_path)
        print(f"Fold {fold+1} 모델 저장 완료: {fold_save_path}")

        if fold == 0:  # 첫 fold 모델로 Feature Importance 기록
            fi = pd.DataFrame({
                'feature': X.columns,
                'importance': model.get_feature_importance()
            }).sort_values("importance", ascending=False)

    # ==========================
    # CV 결과 출력
    # ==========================
    print("\n========== CV RESULT ==========")
    print("Mean Accuracy:", np.mean(eval_metrics_dict['accuracy']))
    print("Mean AUC:", np.mean(eval_metrics_dict['roc_auc']))
    print("Mean F1 :", np.mean(eval_metrics_dict['f1_score']))
    print("Mean Precision:", np.mean(eval_metrics_dict['precision']))
    print("Mean Recall:", np.mean(eval_metrics_dict['recall']))

    # ==========================
    # Test 예측 합치기
    # ==========================
    test_pred_final = np.mean(test_pred_list, axis=0)

    # ==========================
    # OOF & Test 저장
    # ==========================
    np.save(os.path.join(save_dir, f"catboost_oofdrop12.npy"), oof_pred)
    np.save(os.path.join(save_dir, f"catboost_test_preddrop12.npy"), test_pred_final)

    # ==========================
    # 제출용 CSV
    # ==========================
    submission = pd.read_csv("./open/sample_submission.csv")
    submission.iloc[:, 1] = test_pred_final
    submission_file = f'osm_{datetime.now().strftime("%m%d-%H%M")}.csv'
    submission.to_csv(submission_file, index=False)
    print(f"\n✅ 제출 파일 생성 완료: {submission_file}")

    return model, oof_pred, test_pred_final, fi

# ==========================
# 함수 실행 예시
# ==========================
model, oof_pred, test_pred_final, fi = run_catboost_cv(
    train_reduced,      # 기존 train -> train_reduced
    test_reduced,       # 기존 test  -> test_reduced
    TARGET,
    cat_features=None,  # 자동으로 object 컬럼 찾아서 cat_features 지정
    n_splits=25,
    threshold=0.5,
    save_dir="./models/drop10",
    random_state=2024,
    verbose=100
)


========== Fold 1 ==========
0:	learn: 0.6610701	test: 0.6611901	best: 0.6611901 (0)	total: 263ms	remaining: 13m 7s
100:	learn: 0.4870408	test: 0.4866974	best: 0.4866974 (100)	total: 20.6s	remaining: 9m 50s
200:	learn: 0.4846382	test: 0.4853979	best: 0.4853935 (199)	total: 41s	remaining: 9m 31s
300:	learn: 0.4828618	test: 0.4850564	best: 0.4850410 (294)	total: 1m 2s	remaining: 9m 20s
400:	learn: 0.4814278	test: 0.4849972	best: 0.4849804 (394)	total: 1m 24s	remaining: 9m 4s
500:	learn: 0.4800389	test: 0.4849818	best: 0.4848943 (470)	total: 1m 45s	remaining: 8m 46s
600:	learn: 0.4787846	test: 0.4849838	best: 0.4848943 (470)	total: 2m 7s	remaining: 8m 27s
700:	learn: 0.4775889	test: 0.4849795	best: 0.4848943 (470)	total: 2m 29s	remaining: 8m 8s
Stopped by overfitting detector  (250 iterations wait)

bestTest = 0.4848942508
bestIteration = 470

Shrink model to first 471 iterations.
Fold 1 모델 저장 완료: ./models/drop10/CatBoost_Fold1.cbm

========== Fold 2 ==========
0:	learn: 0.6588992	test: 

In [21]:
print("Mean Accuracy:", np.mean(eval_metrics_dict['accuracy']))

Mean Accuracy: 0.7464921159544164


# TabNet

In [29]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
import numpy as np

TARGET = "임신 성공 여부"

train_tab = train.copy()
test_tab = test.copy()

# ==============================
# 1. object → 숫자
# ==============================
obj_cols = train_tab.select_dtypes(include=["object"]).columns.tolist()

if len(obj_cols) > 0:
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    train_tab[obj_cols] = enc.fit_transform(train_tab[obj_cols])
    test_tab[obj_cols] = enc.transform(test_tab[obj_cols])

# ==============================
# 2. 결측
# ==============================
train_tab = train_tab.fillna(-1)
test_tab = test_tab.fillna(-1)

# ==============================
# 3. 타입 정리
# ==============================
for col in train_tab.columns:
    if col == TARGET:
        continue
    train_tab[col] = train_tab[col].astype(np.float32)
    test_tab[col] = test_tab[col].astype(np.float32)

# ==============================
# 4. 🔥 타겟 분리 (여기가 핵심)
# ==============================
y = train_tab[TARGET].values

X_df = train_tab.drop(columns=[TARGET])
X_test_df = test_tab.copy()

# ==============================
# 5. Scaling (feature만!)
# ==============================
scaler = StandardScaler()
X = scaler.fit_transform(X_df)
X_test = scaler.transform(X_test_df)

print("train shape:", X.shape)
print("test shape :", X_test.shape)
print("nan check:", np.isnan(X).sum(), np.isnan(X_test).sum())


train shape: (256351, 120)
test shape : (90067, 120)
nan check: 0 0


In [30]:
# ==============================
# TabNet KFold CV (앙상블용)
# ==============================
import os, gc, random
import numpy as np
import torch

from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score,
    f1_score, roc_auc_score
)

SEED = 2024
N_SPLITS = 5          # ← 25는 너무 오래걸림. 필요하면 나중에 늘리자.
TH = 0.5
SAVE_DIR = "./models/tabnet"
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==============================
# 기록용
# ==============================
eval_metrics_dict = {k: [] for k in [
    'confusion','accuracy','precision','recall','f1_score','roc_auc'
]}

tabnet_oof_pred = np.zeros(len(y))
tabnet_test_preds = []

# ==============================
# KFold
# ==============================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    print(f"\n========== TabNet Fold {fold} ==========")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = TabNetClassifier(
        n_d=64,                    # 표현력 증가
        n_a=64,
        n_steps=5,
        gamma=1.5,                 # feature reuse ↑
        lambda_sparse=1e-4,        # 과적합 방지 핵심
        optimizer_fn=torch.optim.AdamW,
        optimizer_params=dict(lr=8e-4, weight_decay=1e-4),
        mask_type='entmax',
        scheduler_params={"step_size": 25, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        seed=SEED,
        verbose=0
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=['auc'],
        max_epochs=80,             # 대용량 = 길게 필요 없음
        patience=15,
        batch_size=4096,           # ★ 매우 중요
        virtual_batch_size=512
    )

    # ==============================
    # validation
    # ==============================
    val_pred_proba = model.predict_proba(X_val)[:, 1]
    val_pred = (val_pred_proba >= TH).astype(int)

    tabnet_oof_pred[val_idx] = val_pred_proba

    eval_metrics_dict['confusion'].append(confusion_matrix(y_val, val_pred))
    eval_metrics_dict['accuracy'].append(accuracy_score(y_val, val_pred))
    eval_metrics_dict['precision'].append(precision_score(y_val, val_pred))
    eval_metrics_dict['recall'].append(recall_score(y_val, val_pred))
    eval_metrics_dict['f1_score'].append(f1_score(y_val, val_pred))
    eval_metrics_dict['roc_auc'].append(roc_auc_score(y_val, val_pred_proba))

    print(f"Fold {fold} | AUC {eval_metrics_dict['roc_auc'][-1]:.4f}")

    # ==============================
    # test
    # ==============================
    tabnet_test_preds.append(model.predict_proba(X_test)[:, 1])

    # 모델 저장
    model.save_model(os.path.join(SAVE_DIR, f"tabnet_fold{fold}"))
    
    del model
    torch.cuda.empty_cache()
    gc.collect()

# ==============================
# CV 평균
# ==============================
print("\n========== TabNet CV RESULT ==========")
print("Mean AUC:", np.mean(eval_metrics_dict['roc_auc']))
print("Mean F1 :", np.mean(eval_metrics_dict['f1_score']))
print("Mean accuracy :", np.mean(eval_metrics_dict['accuracy']))

# ==============================
# test 평균
# ==============================
tabnet_test_pred_final = np.mean(tabnet_test_preds, axis=0)

# ==============================
# 저장 (앙상블용 핵심!)
# ==============================
np.save(os.path.join(SAVE_DIR, "tabnet_oof_proba.npy"), tabnet_oof_pred)
np.save(os.path.join(SAVE_DIR, "tabnet_test_proba.npy"), tabnet_test_pred_final)

print("✅ TabNet OOF & Test 저장 완료")



========== TabNet Fold 1 ==========
Stop training because you reached max_epochs = 80 with best_epoch = 73 and best_val_0_auc = 0.73643
Fold 1 | AUC 0.7364
Successfully saved model at ./models/tabnet/tabnet_fold1.zip

========== TabNet Fold 2 ==========
Stop training because you reached max_epochs = 80 with best_epoch = 78 and best_val_0_auc = 0.73279
Fold 2 | AUC 0.7328
Successfully saved model at ./models/tabnet/tabnet_fold2.zip

========== TabNet Fold 3 ==========
Stop training because you reached max_epochs = 80 with best_epoch = 77 and best_val_0_auc = 0.7329
Fold 3 | AUC 0.7329
Successfully saved model at ./models/tabnet/tabnet_fold3.zip

========== TabNet Fold 4 ==========
Stop training because you reached max_epochs = 80 with best_epoch = 74 and best_val_0_auc = 0.73285
Fold 4 | AUC 0.7329
Successfully saved model at ./models/tabnet/tabnet_fold4.zip

========== TabNet Fold 5 ==========

Early stopping occurred at epoch 47 with best_epoch = 32 and best_val_0_auc = 0.72519
Fold 

# 블렌딩

In [31]:
from itertools import combinations
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

TH = 0.5

cat_oof = np.load("./models/catboost_oof.npy")
cat_test = np.load("./models/IVF_test_pred_mean.npy")

tab_oof = np.load("./models/tabnet/tabnet_oof_proba.npy")
tab_test = np.load("./models/tabnet/tabnet_test_proba.npy")

nn_oof = np.load("nn_oof.npy")
nn_test = np.load("nn_test.npy")

# ======================================================
# 모델 등록
# ======================================================
oof_dict = {
    "cat": cat_oof,
    "tab": tab_oof,
    "nn": nn_oof,
}

test_dict = {
    "cat": cat_test,
    "tab": tab_test,
    "nn": nn_test,
}

model_names = list(oof_dict.keys())

best_auc = -1
best_f1 = -1
best_acc = -1
best_weights = None
best_models = None

# ======================================================
# 조합 탐색
# ======================================================
for r in range(2, len(model_names) + 1):
    for comb in combinations(model_names, r):

        print(f"\n조합:", comb)

        for _ in range(3000):
            ws = np.random.dirichlet(np.ones(len(comb)), size=1)[0]

            blend = np.zeros(len(y))
            for i, m in enumerate(comb):
                blend += ws[i] * oof_dict[m]

            auc = roc_auc_score(y, blend)
            pred = (blend >= TH).astype(int)
            f1 = f1_score(y, pred)
            acc = accuracy_score(y, pred)

            if auc > best_auc:
                best_auc = auc
                best_f1 = f1
                best_acc = acc
                best_weights = ws
                best_models = comb

print("\n==============================")
print("🔥 BEST RESULT")
print("models :", best_models)
print("weights:", best_weights)
print("AUC :", best_auc)
print("F1  :", best_f1)
print("ACC :", best_acc)

# ======================================================
# Test 적용
# ======================================================
blend_test_pred = np.zeros(len(list(test_dict.values())[0]))

for i, m in enumerate(best_models):
    blend_test_pred += best_weights[i] * test_dict[m]

np.save(".blend_multi_test_pred.npy", blend_test_pred)
print("✅ multi blend test 저장 완료")



조합: ('cat', 'tab')

조합: ('cat', 'nn')

조합: ('tab', 'nn')

조합: ('cat', 'tab', 'nn')

🔥 BEST RESULT
models : ('cat', 'tab', 'nn')
weights: [0.89386515 0.04152414 0.06461071]
AUC : 0.740909864300151
F1  : 0.2408930258599573
ACC : 0.7461332313897742


ValueError: non-broadcastable output operand with shape (90067,) doesn't match the broadcast shape (90067,90067)

In [25]:
print(len(y), len(cat_oof), len(tab_oof), len(nn_oof))

256351 256351 256351 256351


In [26]:
print(np.isnan(cat_oof).sum(),
    np.isnan(tab_oof).sum(),
    np.isnan(nn_oof).sum())

0 0 0


In [27]:
for name, pred in oof_dict.items():
    auc = roc_auc_score(y, pred)
    print(name, auc)


cat 0.7408575639963026
tab 0.7363262559023425
nn 0.737687456531412
